# Fine-Tuning and Low-Rank Adaptation

A pretrained encoder can be **read** (a probe on frozen features), **rewritten** (full fine-tuning), or **nudged**: each
weight matrix moves by a rank-$r$ product, $W = W_0 + s\,AB$ (LoRA). The encoder is Topic 12's filing encoder
($V = 879$, $n = 32$, $d = 64$, two pre-LN blocks, 158,447 parameters, pretrained from scratch by masked-LM); the task
is Topic 19's aspect sentiment, read from the mean of the encoder's final states over the aspect span. This notebook
makes seven points:
- the factored gradient is $\partial L/\partial A = sGB^\top$, $\partial L/\partial B = sA^\top G$, so from $B = 0$ the
  first step is the full step pushed through the random map $A_0A_0^\top$, rank-stable only when $s \propto 1/\sqrt r$
  (Thm 1);
- gradient flow conserves $A^\top A - BB^\top$ (Thm 2), and Adam does not;
- full fine-tuning's update is effectively low-rank here (Prop 2), though LoRA does not find the rank-4 solution;
- SGD moves a linear layer only along its training inputs, so unseen tokens never move, and a probe-first start
  distorts less (Thm 3, Kumar et al.); L2-SP is a Gaussian prior at $\theta_0$ (Prop 3);
- a random subspace step is unbiased (Prop 4), and the pretrained encoder learns more in every subspace;
- fine-tuning a linearized model is kernel gradient descent (Prop 5), which GD from the probe's head follows and
  Adam leaves;
- the encoder reads negation but learns a shortcut, which counterfactual rows remove; read aspect by aspect, the
  filings give up their tone.

Run from the repository root (≈ 2½ min; the encoder pretrains in ≈ 21 s):
`uv run --with numpy==2.5.4 --with scipy==1.18.1 --with jupyter --with ipykernel jupyter execute notebooks/fine-tuning-low-rank-adaptation/01_fine_tuning_low_rank_adaptation.ipynb`

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "fine_tuning_low_rank_adaptation.py").exists()
                       else here / "notebooks" / "fine-tuning-low-rank-adaptation"))
import fine_tuning_low_rank_adaptation as ft

cm, ab = ft.cm, ft.ab
tr = ft.pretrained()
n_par = sum(v.size for k, v in tr.P.items() if k != "b") + sum(v.size for b in tr.blocks for s in b for v in b[s].values())
print("encoder parameters (without the masked-LM bias):", n_par, " vocabulary:", cm.filing_vocab()[1])
enc, y = ft.train_set(False)
print("training rows:", len(enc), " counterfactual rows:", len(ft.augmented_rows()), " test rows:",
      len(ft.eval_set("test")[0]))
print("held-out masked-LM loss of the pretrained encoder:", round(ft.mlm_loss(*ft.base("pretrained")), 6))

## 1. Read, rewrite, or nudge

The probe fits Topic 17's logistic regression on frozen span features. Full fine-tuning frees every parameter (Adam,
the rate decaying linearly to 0, three epochs). The never-pretrained twin is the same architecture at Topic 12's
initialization, fine-tuned identically: the gap between it and the pretrained encoder is what pretraining buys.

In [ ]:
rows = [("probe", ft.evaluate(None)), ("random-init, fine-tuned", ft.evaluate(ft.RANDOM_RUN))]
rows += [(f"full fine-tuning, seed {c.seed}", ft.evaluate(c)) for c in ft.HEADLINE["full"]]
for name, e in rows:
    print(f"{name:28} acc {e['acc']:.6f}  negated adjective {e['by']['negated adjective']:.6f}  "
          f"masked-LM loss {e['mlm']:.6f}")

## 2. One weight, two factors (Def 1, Prop 1)

$W = W_0 + s\,AB$ with $A \in \mathbb{R}^{k\times r}$ Gaussian ($\sigma^2 = 1/k$) and $B \in \mathbb{R}^{r\times m}$
zero, so the update starts at 0. It has rank at most $r$ and $r(k+m)$ parameters instead of $km$; after training it
merges into one matrix; and $(AS, S^{-1}B)$ gives the same update for every invertible $S$. In one dimension,
$w = s\,ab$: the reachable updates are the level sets $ab = \text{const}$, hyperbolas in the $(a, b)$ plane.

In [ ]:
for a0 in ft.SCALAR_A0:
    for opt in ("gd", "adam"):
        p = ft.scalar_path(a0, ft.SCALAR_ETA[0], opt)
        inv = p[:, 0] ** 2 - p[:, 1] ** 2
        print(f"a0 = {a0:4}  {opt:4}  final (a, b) = ({p[-1, 0]:.4f}, {p[-1, 1]:.4f})  ab = {p[-1, 0] * p[-1, 1]:.4f}"
              f"  max |a² − b² − a0²| = {np.abs(inv - a0**2).max():.4f}")

## 3. The first step (Thm 1)

With $G = \partial L/\partial W$, Thm 1(a) gives $\partial L/\partial A = sGB^\top = 0$ at $B = 0$ and
$\partial L/\partial B = sA^\top G$. One SGD step therefore updates $W$ by $\Delta W_1 = -\eta s^2 A_0A_0^\top G$: the
full step pushed through a random rank-$r$ map whose mean is $r\sigma^2 I$ (Thm 1(c)), so $\mathbb{E}\,\Delta W_1 =
-\eta s^2 r\sigma^2 G$, independent of $r$ only when $s \propto 1/\sqrt r$ (rsLoRA). A single draw is noisy:
$\mathbb{E}\lVert A_0A_0^\top - r\sigma^2 I\rVert_F^2 = r\sigma^4 k(k+1)$, a relative $\sqrt{(k+1)/r}$. Adam's first
step is a sign step, $\Delta W_1 = -\eta s A_0\,\mathrm{sgn}(A_0^\top G)$, whose norm lies between
$\eta s\sqrt{rm}\,\sigma_{\min}(A_0)$ and $\eta s\sqrt{rm}\,\sigma_{\max}(A_0)$ (Thm 1(d)): rank-stable again only
under $\alpha/\sqrt r$. Below, the four adapted matrices of the encoder, from the probe's head (at a zero head every
encoder gradient is zero).

In [ ]:
fs = ft.first_step_table()
print("r:", ft.FIRST_R)
for key in (("adam", "sqrt"), ("adam", "lin"), ("sgd", "sqrt"), ("sgd", "lin")):
    print(f"{key[0]:4} {'α/√r' if key[1] == 'sqrt' else 'α/r ':4}", " ".join(f"{v:.6f}" for v in fs[key]))
print("mean SGD step under α/√r:", round(fs["expected_sgd"]["sqrt"][0], 6))
print("relative fluctuation √((k+1)/r) at k = 64:", [round(math.sqrt(65 / r), 3) for r in ft.FIRST_R])

Trained to the end, the scaling decides the update's size: under $\alpha/\sqrt r$ the merged update grows with
$r$, under $\alpha/r$ it shrinks. Neither wins on accuracy at a fixed rate here; what rank stability promises is that
the right rate does not move with $r$.

In [ ]:
for (r, sc), c in ft.RANK_RUNS.items():
    e = ft.evaluate(ft._canon(c))
    print(f"r = {r:2}  {sc:4}  acc {e['acc']:.6f}  masked-LM loss {e['mlm']:.6f}  ‖sAB‖ {ft.trained_lora_norm(c):.6f}")

## 4. What two factors conserve (Thm 2)

Under gradient flow, $\frac{d}{dt}(A^\top A) = \frac{d}{dt}(BB^\top) = -s(A^\top GB^\top + BG^\top A)$, so
$A^\top A - BB^\top = A_0^\top A_0$ for all time (Du, Hu, and Lee; Arora, Cohen, and Hazan): $A$ must grow as $B$
grows. One GD step changes it by exactly $\eta^2 s^2(BG^\top GB^\top - A^\top GG^\top A)$, so at a fixed horizon the
drift is $O(\eta)$. Adam rescales each coordinate and breaks it.

In [ ]:
td = ft.toy_drift()
print("toy drift at horizon t = 8:", [f"{d:.3e}" for d in td["drift"]], " slope in η:", round(td["slope"], 6))
print("headline LoRA (Adam) relative drift:", round(ft.balance_drift(ft.HEADLINE["lora"][0]), 6))
n = ft.ntk_experiment()
print("from the LP-FT start: GD drift", f"{n['gd']['balance_drift']:.2e}", " Adam drift", round(n["adam"]["balance_drift"], 6))

## 5. Is the update low-rank? (Prop 2)

The best rank-$r$ approximation of an update is its truncated SVD (Eckart–Young–Mirsky, formalML `svd` Thm 2). We
compare each matrix's $\sigma_1$ with the edge $\tau(\sqrt k + \sqrt m)$ of a Gaussian matrix of the same Frobenius
norm, and merge truncated updates back into the fine-tuned model.

In [ ]:
for x in ft.spectra():
    print(f"block {x['block']} {x['matrix']:2}  σ1/edge {x['sv'][0] / x['edge']:.3f}  stable rank {x['stable_rank']:.3f}"
          f"  top-4 energy {x['top4']:.3f}")
t = ft.truncations()
print("truncated-merge accuracy:", dict(zip(t["r"], [round(a, 6) for a in t["acc"]])), " full:", round(t["full"], 6))
print("LoRA trained at r = 4:", {sc: round(ft.evaluate(ft._canon(ft.RANK_RUNS[(4, sc)]))["acc"], 6) for sc in ("lin", "sqrt")})

## 6. What fine-tuning leaves alone (Thm 3, Prop 3)

A linear layer on inputs $x_i$ gets gradient $\sum_i x_i^\top\delta_i$, so SGD moves it only along the span of its
training inputs (Thm 3(a)); Adam's per-coordinate scaling breaks that, but a row whose gradient is always zero keeps
zero moments and never moves (Thm 3(b)). In a two-layer linear network the features move only on the training span,
so starting from the probe's head on perfect features leaves the out-of-distribution loss at 0, while a random head
does not (Kumar et al.'s Prop 3.7). Forgetting is the rise of held-out masked-LM loss.

In [ ]:
print("span residual: SGD", f"{ft.span_toy('sgd'):.2e}", " Adam", round(ft.span_toy("adam"), 6))
print("unseen tokens:", len(ft.unseen_tokens()), " rows moved (all, unseen): full", ft.embedding_rows_moved(ft.HEADLINE["full"][0]),
      " LoRA", ft.embedding_rows_moved(ft.HEADLINE["lora"][0]))
print("Kumar toy OOD loss: LP-FT", f"{ft.kumar_toy('lp')['ood']:.2e}", " random head", round(ft.kumar_toy("random")["ood"], 6))
for m in ("full", "lpft", "lora"):
    print(f"{m:5} acc mean/min/max", [round(v, 6) for v in ft.spread(m)], " masked-LM", [round(v, 6) for v in ft.spread(m, "mlm")])

L2-SP adds $\tfrac{\lambda}{2}\lVert\theta - \theta_0\rVert^2$: MAP under $N(\theta_0, \lambda^{-1}I)$
(formalStatistics `regularization-and-penalized-estimation` Thm 5), which near an optimum shrinks the full
displacement by $h/(h + \lambda)$ along each Hessian eigenvector. It trades accuracy for retention; LP-FT gets the
retention without the accuracy cost, and every LP-FT seed beats every LoRA run on both.

In [ ]:
for c in (ft.HEADLINE["full"][0],) + ft.L2SP_RUNS + ft.SLOW_RUNS:
    e = ft.evaluate(c)
    print(f"{c.method:5} λ = {c.lam:<4g} rate {c.lr or ft.LR[c.method]:<7g} acc {e['acc']:.6f}  masked-LM {e['mlm']:.6f}")

## 7. Two lenses on why a small update suffices (Prop 4, Prop 5)

**A random subspace.** $\theta = \theta_0 + Sz$ with a count sketch $S$ (one $\pm 1$ per row) gives
$\mathbb{E}[SS^\top] = I$, so a $z$-step is an unbiased full step (Prop 4). $d_{90}$ is the least dimension reaching
90% of full fine-tuning (Aghajanyan, Gupta, and Zettlemoyer). **A kernel.** Linearizing $f$ around $\theta_0$ makes GD
in $\theta$ exactly $f \leftarrow f - \eta K\nabla_f\ell$ with $K = JJ^\top$ (Prop 5), and at $B = 0$ LoRA's kernel is
the $A_0$-sketched kernel of the adapted matrices (Cor 1). We linearize around the probe's head (a zero head has a
zero Jacobian).

In [ ]:
for i in ("pretrained", "random"):
    d = ft.d90(i)
    print(f"{i:10} sketch acc at D = {ft.SKETCH_DIMS}: {[round(a, 6) for a in d['acc']]}  90% of full: {0.9 * d['full']:.6f}"
          f"  d90: {d['d90']}")
print("start: acc", round(n["acc_start"], 6), " loss", round(n["loss_start"], 6), "| kernel GD: acc", round(n["acc_kernel"], 6),
      " loss", round(n["loss_kernel"], 6))
for opt in ("gd", "adam"):
    print(f"LoRA {opt:4}: acc {n[opt]['acc']:.6f}  loss {n[opt]['loss']:.6f}  agrees with kernel {n[opt]['agree']:.6f}"
          f"  kernel moved {n[opt]['kernel_move']:.6f}")

## 8. Reading negation in filings

Topic 19's negation quadruples swap the adjective of "T was not ADJ this period" and delete "not". The fine-tuned
encoder reads both negated forms but not the deleted-"not" frame, which training never shows without "not": it has
learned the frame. Adding the 310 deleted-"not" training rows (counterfactual augmentation, Kaushik et al.) fixes every
seed. Read aspect by aspect and aggregated by Topic 19's Bayes rule, the filings give up their tone; fine-tuned at the
document level, they do not.

In [ ]:
for name, cs in (("full", ft.HEADLINE["full"]), ("augmented", ft.AUG_RUNS[:3])):
    for c in cs:
        print(f"{name:9} seed {c.seed}  positions (not ADJ+, not ADJ−, ADJ+, ADJ−):",
              [round(v, 4) for v in ft.evaluate(c)["negation"]["pos"]])
for k, v in ft.document_readers().items():
    print(f"{k:28} {v:.6f}")
for i in ("pretrained", "random"):
    print(f"document fine-tune ({i}):", {k: round(v, 6) for k, v in ft.doc_fine_tune(i).items()})
print("aspect fine-tune, aggregated to tone:", round(ft.tone_from(ft.evaluate(ft.HEADLINE["full"][0])["pred"]), 6))

## The whole harness

Every claim above is an assert in the module; this runs them all (the trained models are cached, so it reuses the
runs above).

In [ ]:
assert ft._run_harness() == 0

## Where this sits

Topic 19's one attention layer could not read negation; a fine-tuned contextual encoder does, once its training data
stops letting a frame stand in for the word. The extraction topics (21–22) fine-tune the same kind of encoder for span
labeling. formalML's `svd` proves the Eckart–Young–Mirsky theorem behind Prop 2 and its `representation-learning`
defines the linear probe; formalStatistics' `regularization-and-penalized-estimation` is L2-SP's ridge;
formalRAG's `dense-retrieval-dual-encoders` bounds the rank of a dual encoder's *scores*, where this topic bounds the
rank of an *update*.